# 01 — Auditoría de calidad de datos (A1)
Objetivo: demostrar que los datos de RetailVenture son confiables antes de analizarlos.
El notebook tiene tres bloques:
1. Perfil general: ¿cada tabla está sana por sí sola?
2. Integridad y reglas de negocio: ¿las tablas son coherentes entre sí y con el negocio?
3. Métricas de calidad y exportación de tablas limpias.

## Bloque 1 — Perfil general
Para cada tabla: filas, columnas, nulos, filas duplicadas, claves primarias duplicadas
y rango de fechas.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
CLEAN = Path("../data/clean")
CLEAN.mkdir(parents=True, exist_ok=True)

FECHAS = {
    "clientes": ["fecha_registro"],
    "transacciones": ["fecha"],
    "productos": [],
    "devoluciones": ["fecha_devolucion"],
    "interacciones_crm": ["fecha"],
    "campanas_marketing": ["fecha_envio"],
}
PK = {
    "clientes": "cliente_id",
    "transacciones": "transaccion_id",
    "productos": "producto_id",
    "devoluciones": "devolucion_id",
    "interacciones_crm": "interaccion_id",
    "campanas_marketing": "campana_id",
}

dfs = {n: pd.read_csv(RAW / f"{n}.csv", parse_dates=f) for n, f in FECHAS.items()}

perfil = pd.DataFrame([
    {
        "tabla": n,
        "filas": len(df),
        "columnas": df.shape[1],
        "nulos": int(df.isna().sum().sum()),
        "filas_duplicadas": int(df.duplicated().sum()),
        "pk_duplicadas": int(df[PK[n]].duplicated().sum()),
        "desde": df[FECHAS[n][0]].min().date() if FECHAS[n] else None,
        "hasta": df[FECHAS[n][0]].max().date() if FECHAS[n] else None,
    }
    for n, df in dfs.items()
])
perfil

,tabla,filas,columnas,nulos,filas_duplicadas,pk_duplicadas,desde,hasta
0,clientes,8000,7,0,0,0,2023-01-01,2024-12-31
1,transacciones,50000,10,0,0,0,2023-01-01,2024-12-31
2,productos,318,7,0,0,0,None,None
3,devoluciones,6724,6,0,0,0,2023-01-02,2024-12-31
4,interacciones_crm,4684,7,0,0,0,2023-01-01,2024-12-31
5,campanas_marketing,8820,8,0,0,0,2023-01-01,2024-12-31


**Hallazgo:** cero nulos y cero duplicados en las 6 tablas; los datos cubren
enero 2023 – diciembre 2024. Productos tiene 318 registros, no 320 como dice
el enunciado.